# Few-shot & Output Control
- Few-shot: 원하는 답변 형식을 예시로 보여주고, LLM이 같은 형식으로 답변하도록 유도하는 방법
- Output Control: RAG에서는 답변 내용뿐 아니라 답변 형식도 중요하다. 예를 들어 화면에 표시하거나 API 응답으로 사용하려면 단순 문장보다 구조화된 결과가 유리하다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

LLM_MODEL = 'gpt-4.1-mini'

## 실습용 문서

In [2]:
from langchain_core.documents import Document

DOCS = [
    Document(
        page_content='파리는 프랑스의 수도이며, 주요 관광지로는 에펠탑, 루브르 박물관, 개선문이 있다. 연간 관광객 수는 약 3천만 명으로 알려져 있다.',
        metadata={'doc_id': 'F01', 'city': '파리'}
    ),
    Document(
        page_content='런던은 영국의 수도이며, 주요 관광지로는 버킹엄 궁전, 런던 아이, 타워 브릿지가 있다. 연간 관광객 수는 약 2천만 명으로 알려져 있다.',
        metadata={'doc_id': 'F02', 'city': '런던'}
    ),
    Document(
        page_content='교토는 일본의 옛 수도이며, 주요 관광지로는 금각사, 은각사, 기요미즈데라가 있다. 연간 관광객 수는 약 1천5백만 명으로 알려져 있다.',
        metadata={'doc_id': 'F03', 'city': '교토'}
    ),
]

def fake_retriever(query: str, k: int = 3):
    return DOCS[:k]

def format_docs(docs):
    return '\n\n'.join(
        f"[{doc.metadata['doc_id']}] {doc.page_content}"
        for doc in docs
    )

## Few-shot으로 답변 형식 유도하기

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=LLM_MODEL,temperature=0)
output_parser = StrOutputParser()

prompt = PromptTemplate.from_template("""
당신은 도시 정보 문서를 바탕으로 답변하는 RAG assistant입니다.
문서에 있는 내용만 사용하세요.
                                      
아래 예시와 같은 형식으로 답변하세요.

[Example]
질문: 파리의 주요 관광지는 무엇인가요?
답변:
도시: 파리
주요 관광지: 에펠탑, 루브르 박물관, 개선문
참고 문서: F01 

질문: 런던의 주요 관광지는 어디인가요?
답변:
도시: 런던
주요 관광지: 버킹엄 궁전, 런던 아이, 타워 브릿지
참고 문서: F02                                                                                                                                                                                             

[Context]
{context}

[Question]
{question}

[Answer]                                                                                                                                                                                                                                                                                                                    
""")

In [4]:
chain = prompt | llm | output_parser

question = '교토의 주요 관광지는 무엇인가요?'
context = format_docs(fake_retriever(question))

print(chain.invoke({'context':context,'question':question}))

답변:
도시: 교토
주요 관광지: 금각사, 은각사, 기요미즈데라
참고 문서: F03


## 구조화된 출력 사용하기

In [5]:
from pydantic import BaseModel, Field 

class CityAnswer(BaseModel):
    city: str = Field(description='질문에 해당하는 도시 이름')
    answer: str = Field(description='문서 기반 답변')
    places: list[str] = Field(description='주요 관광지 목록')
    source_doc_ids: list[str] = Field(description='답변에 사용한 문서 ID 목록')

structured_llm = llm.with_structured_output(CityAnswer)

structured_prompt = PromptTemplate.from_template("""
다음 문서를 바탕으로 질문에 답하세요.                                                                                                                 
문서에 없는 내용은 추측하지 마세요
                                                 
[Context]
{context}

[Question]
{question}                                                                                                                                                                                                                                                                                       
""")

structured_chain = structured_prompt | structured_llm

result = structured_chain.invoke({
    'context' : context,
    'question' : question
})

result

CityAnswer(city='교토', answer='교토의 주요 관광지는 금각사, 은각사, 기요미즈데라입니다.', places=['금각사', '은각사', '기요미즈데라'], source_doc_ids=['F03'])

## 결과 활용

In [6]:
print(result.answer)
print(result.places)
print(result.city)
print(result.source_doc_ids)

교토의 주요 관광지는 금각사, 은각사, 기요미즈데라입니다.
['금각사', '은각사', '기요미즈데라']
교토
['F03']
